# inf-masking — ex1: causal attention masking with -inf + softmax + viz

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `inf-masking`. Running the final beacon cell reports progress against the `Numpy: Inf-fill masking trick` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import matplotlib.pyplot as plt

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Numpy: Inf-fill masking trick` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`inf-masking`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "inf-masking"
DD_SUBTOPIC = "Numpy: Inf-fill masking trick"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## `masked_fill` with -inf — quick refresher

The canonical attention-masking trick:

```python
scores = scores.masked_fill(mask, float('-inf'))
weights = scores.softmax(dim=-1)
```

**Why -inf and not 0.** Softmax computes `exp(s) / sum(exp(s))`. If you zero out a masked position *before* softmax, `exp(0) = 1` and it still gets weight `1 / (1 + sum_others)` — a non-trivial contribution. If you fill with `-inf`, `exp(-inf) = 0` exactly, so the position contributes zero to both numerator and denominator — fully suppressed.

**Numerical sanity.** PyTorch's softmax does the log-sum-exp trick internally, so even with `-inf` in the input, the output is well-defined (unless an entire row is masked — then you get `nan` from `0/0`).

**In-place vs out-of-place.** `masked_fill_` (trailing underscore) mutates; `masked_fill` returns a new tensor. Both take a `bool` mask with `True` = "fill this position". Mismatched dtype on the mask raises.

**Broadcasting.** The mask broadcasts against the scores tensor. A `(T, T)` causal mask applied to `(B, H, T, T)` scores is the standard transformer pattern — the mask is replicated across batch and heads with zero memory cost.

### Exercise 1 — causal attention masking with -inf + softmax + viz

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Apply `masked_fill` with `-inf` to a scores tensor before softmax so the masked positions receive zero weight in the output distribution, then visualise before-and-after attention heatmaps to confirm the masked half is fully suppressed.
> Keywords: attention, causal-mask, softmax, visualization
> ```

**KCs targeted:** `masked-fill-neg-inf-before-softmax`, `softmax-zeros-masked-positions`

Implement `ex1_causal_softmax(scores)`.

- `scores` has shape `(T, T)` — pre-softmax attention logits.
- Build a causal mask: position `i` may attend to positions `0..i` but not `i+1..T-1`. (True above the diagonal = blocked.)
- Fill blocked positions with `-inf`, then softmax along the **last axis** (the key axis).
- Return the `(T, T)` attention weights.

**Why -inf and not 0.** `softmax(0) = exp(0) / sum_exp = 1 / (1 + sum_others)` — masked positions still get a tiny non-zero weight. `softmax(-inf) = exp(-inf) / sum_exp = 0 / sum_exp = 0` exactly. Verifying this is the whole point of the exercise.

The visualization renders the *before-mask* and *after-mask* softmax side by side so you can see the upper triangle go from non-zero to flat zero.

In [ ]:
def ex1_causal_softmax(scores: Tensor) -> Tensor:
    T = scores.shape[-1]
    mask = t.triu(t.ones(T, T, dtype=t.bool), diagonal=1)
    return scores.masked_fill(mask, float('-inf')).softmax(dim=-1)


<details><summary>Solution</summary>

```python
def ex1_causal_softmax(scores: Tensor) -> Tensor:
    T = scores.shape[-1]
    mask = t.triu(t.ones(T, T, dtype=t.bool), diagonal=1)
    return scores.masked_fill(mask, float('-inf')).softmax(dim=-1)
```

**`t.triu(..., diagonal=1)`.** Returns the strict upper triangle (excluding the diagonal). For causal attention, the diagonal *must* be allowed (a token can attend to itself), so use `diagonal=1` not `diagonal=0`.

**Why `masked_fill` not `scores[mask] = -inf`.** Same result, but `masked_fill` is functional (returns a new tensor — no surprise mutation of the caller's `scores`) and is the idiomatic torch API for this exact pattern.

**Row-of-all-masked → nan.** If an entire row is masked (no valid keys at all), softmax does `0/0` and you get `nan`. In practice the diagonal is always unmasked for causal attention, so the issue doesn't arise — but for arbitrary masks, add a sentinel: ensure at least one True per row before applying.

**Numerical stability.** PyTorch's `softmax` does log-sum-exp internally: it subtracts `max(scores, dim=-1)` before exping. If your row has `-inf`s, the subtraction yields `-inf` minus a finite number = `-inf`, so `exp(...) = 0` exactly. The math stays clean — no `inf - inf = nan` traps.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()